In [1]:
from presyo_ph.medallion.silver import load_silver
from presyo_ph.spark import get_spark_session
from presyo_ph.medallion.gold import save_gold_tables
from pyspark.sql import functions as F

In [ ]:
spark = get_spark_session()
silver_df = load_silver(spark)

# Inspect Schema

In [3]:
silver_df.printSchema()

root
 |-- ISO3: string (nullable = true)
 |-- country: string (nullable = true)
 |-- adm1_name: string (nullable = true)
 |-- adm2_name: string (nullable = true)
 |-- mkt_name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- geo_id: string (nullable = true)
 |-- price_date: date (nullable = true)
 |-- year: double (nullable = true)
 |-- month: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- components: string (nullable = true)
 |-- start_dense_data: date (nullable = true)
 |-- last_survey_point: date (nullable = true)
 |-- data_coverage: double (nullable = true)
 |-- data_coverage_recent: double (nullable = true)
 |-- index_confidence_score: double (nullable = true)
 |-- spatially_interpolated: boolean (nullable = true)
 |-- beans: double (nullable = true)
 |-- cabbage: double (nullable = true)
 |-- carrots: double (nullable = true)
 |-- eggs: double (nullable = true)
 |-- garlic: double (nullable = true)
 |-- on

# Define commodities

In [4]:
COMMODITY_COLUMNS = [
    "beans",
    "cabbage",
    "carrots",
    "eggs",
    "garlic",
    "onions",
    "potatoes",
    "rice",
    "tomatoes",
]

# Define market monthly prices

In [5]:
commodity_records = [
    F.struct(
        F.lit(commodity).alias("commodity"),
        F.col(f"{commodity}_filled").alias("price"),
        F.col(commodity).alias("original_price"),
        F.col(f"{commodity}_was_imputed").alias("was_imputed"),
    )
    for commodity in COMMODITY_COLUMNS
]

market_prices_df = (
    silver_df
    .select(
        "geo_id",
        "mkt_name",
        "adm1_name",
        "adm2_name",
        "price_date",
        "currency",
        F.explode(F.array(*commodity_records)).alias("price_record"),
    )
    .select(
        "geo_id",
        "mkt_name",
        "adm1_name",
        "adm2_name",
        "price_date",
        "currency",
        F.col("price_record.commodity").alias("commodity"),
        F.col("price_record.price").alias("price"),
        F.col("price_record.original_price").alias("original_price"),
        F.col("price_record.was_imputed").alias("was_imputed"),
    )
)

print(f"Market price records: {market_prices_df.count():,}")
market_prices_df.limit(10).toPandas()

26/11/05 14:46:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Market price records: 222,687


,geo_id,mkt_name,adm1_name,adm2_name,price_date,currency,commodity,price,original_price,was_imputed
0,gid_1014000001250000000,Catarman,Region VIII,Southern Leyte,2007-01-01,PHP,beans,NaN,NaN,False
1,gid_1014000001250000000,Catarman,Region VIII,Southern Leyte,2007-01-01,PHP,cabbage,NaN,NaN,False
2,gid_1014000001250000000,Catarman,Region VIII,Southern Leyte,2007-01-01,PHP,carrots,NaN,NaN,False
3,gid_1014000001250000000,Catarman,Region VIII,Southern Leyte,2007-01-01,PHP,eggs,NaN,NaN,False
4,gid_1014000001250000000,Catarman,Region VIII,Southern Leyte,2007-01-01,PHP,garlic,NaN,NaN,False
5,gid_1014000001250000000,Catarman,Region VIII,Southern Leyte,2007-01-01,PHP,onions,NaN,NaN,False
6,gid_1014000001250000000,Catarman,Region VIII,Southern Leyte,2007-01-01,PHP,potatoes,NaN,NaN,False
7,gid_1014000001250000000,Catarman,Region VIII,Southern Leyte,2007-01-01,PHP,rice,NaN,NaN,False
8,gid_1014000001250000000,Catarman,Region VIII,Southern Leyte,2007-01-01,PHP,tomatoes,NaN,NaN,False
9,gid_1014000001250000000,Catarman,Region VIII,Southern Leyte,2007-02-01,PHP,beans,NaN,NaN,False


# Regional Prices
Regions are not weighted

In [6]:
regional_prices_df = (
    market_prices_df
    .filter(F.col("price").isNotNull())
    .groupBy(
        "adm1_name",
        "price_date",
        "commodity",
        "currency",
    )
    .agg(
        F.avg("price").alias("avg_price"),
        F.expr("percentile_approx(price, 0.5)").alias("median_price"),
        F.min("price").alias("min_price"),
        F.max("price").alias("max_price"),
        F.countDistinct("geo_id").alias("markets_reporting"),
        F.sum(
            F.col("was_imputed").cast("integer")
        ).alias("imputed_observations"),
    )
)

print(f"Regional records: {regional_prices_df.count():,}")
regional_prices_df.limit(10).toPandas()

Regional records: 23,862


,adm1_name,price_date,commodity,currency,avg_price,median_price,min_price,max_price,markets_reporting,imputed_observations
0,Autonomous region in Muslim Mindanao,2008-01-01,onions,PHP,148.46,148.46,148.46,148.46,1,0
1,Autonomous region in Muslim Mindanao,2008-02-01,garlic,PHP,155.08,155.08,155.08,155.08,1,0
2,Autonomous region in Muslim Mindanao,2008-02-01,onions,PHP,86.46,86.46,86.46,86.46,1,0
3,Autonomous region in Muslim Mindanao,2008-02-01,tomatoes,PHP,30.85,30.85,30.85,30.85,1,0
4,Autonomous region in Muslim Mindanao,2008-03-01,cabbage,PHP,26.54,26.54,26.54,26.54,1,1
5,Autonomous region in Muslim Mindanao,2008-03-01,carrots,PHP,44.42,44.42,44.42,44.42,1,0
6,Autonomous region in Muslim Mindanao,2008-03-01,garlic,PHP,112.00,112.00,112.00,112.00,1,0
7,Autonomous region in Muslim Mindanao,2008-03-01,tomatoes,PHP,22.18,22.18,22.18,22.18,1,0
8,Autonomous region in Muslim Mindanao,2008-04-01,carrots,PHP,56.08,56.08,56.08,56.08,1,0
9,Autonomous region in Muslim Mindanao,2008-04-01,garlic,PHP,112.00,112.00,112.00,112.00,1,0


# National Prices
Locations are not weighted

In [7]:
national_prices_df = (
    market_prices_df
    .filter(F.col("price").isNotNull())
    .groupBy(
        "price_date",
        "commodity",
        "currency",
    )
    .agg(
        F.avg("price").alias("avg_price"),
        F.expr("percentile_approx(price, 0.5)").alias("median_price"),
        F.min("price").alias("min_price"),
        F.max("price").alias("max_price"),
        F.countDistinct("geo_id").alias("markets_reporting"),
        F.sum(
            F.col("was_imputed").cast("integer")
        ).alias("imputed_observations"),
    )
)

print(f"National records: {national_prices_df.count():,}")
national_prices_df.limit(10).toPandas()

National records: 1,620


,price_date,commodity,currency,avg_price,median_price,min_price,max_price,markets_reporting,imputed_observations
0,2007-01-01,rice,PHP,21.402857,21.19,20.00,22.88,7,0
1,2007-02-01,rice,PHP,21.645714,21.26,20.25,23.00,7,0
2,2007-03-01,rice,PHP,22.217143,22.80,20.20,23.00,7,0
3,2007-04-01,rice,PHP,22.410714,23.00,20.00,23.38,7,1
4,2007-05-01,rice,PHP,22.712857,23.00,20.00,24.00,7,0
5,2007-06-01,rice,PHP,22.937143,23.60,20.60,24.10,7,0
6,2007-07-01,rice,PHP,23.484286,23.88,21.00,25.25,7,0
7,2007-08-01,rice,PHP,23.911429,24.13,22.00,25.50,7,0
8,2007-09-01,rice,PHP,23.857143,24.00,22.67,25.00,7,0
9,2007-10-01,rice,PHP,23.830000,24.00,22.75,25.00,7,0


# National Commodity Price Growth

In [8]:
national_commodity_growth_df = (
    market_prices_df
    .filter(F.col("price").isNotNull())
    .groupBy("commodity", "currency")
    .agg(
        F.min("price_date").alias("start_date"),
        F.max("price_date").alias("end_date"),
    )
)

print(f"National commodity price growth records: {national_commodity_growth_df.count():,}")
national_commodity_growth_df.toPandas()

National commodity price growth records: 9


,commodity,currency,start_date,end_date
0,beans,PHP,2012-05-01,2025-09-01
1,cabbage,PHP,2008-01-01,2025-09-01
2,potatoes,PHP,2008-01-01,2025-09-01
3,tomatoes,PHP,2008-01-01,2025-09-01
4,carrots,PHP,2008-01-01,2025-09-01
5,garlic,PHP,2008-01-01,2025-09-01
6,rice,PHP,2007-01-01,2025-09-01
7,eggs,PHP,2008-01-01,2025-09-01
8,onions,PHP,2008-01-01,2025-09-01


In [9]:
start_prices = (
    market_prices_df
    .filter(F.col("price").isNotNull())
    .groupBy("commodity", "currency", "price_date")
    .agg(
        F.avg("price").alias("start_price")
    )
)

end_prices = (
    market_prices_df
    .filter(F.col("price").isNotNull())
    .groupBy("commodity", "currency", "price_date")
    .agg(
        F.avg("price").alias("end_price")
    )
)

In [10]:
national_commodity_growth_df = (
    market_prices_df
    .filter(F.col("price").isNotNull())
    .groupBy("commodity", "currency")
    .agg(
        F.min("price_date").alias("start_date"),
        F.max("price_date").alias("end_date"),
    )
    .join(
        market_prices_df
        .filter(F.col("price").isNotNull())
        .groupBy("commodity", "currency")
        .agg(
            F.min("price_date").alias("start_date"),
            F.max("price_date").alias("end_date"),
        ),
        ["commodity", "currency", "start_date", "end_date"],
        "inner",
    )
)

In [11]:
date_ranges = (
    market_prices_df
    .filter(F.col("price").isNotNull())
    .groupBy("commodity", "currency")
    .agg(
        F.min("price_date").alias("start_date"),
        F.max("price_date").alias("end_date"),
    )
)

prices = market_prices_df.alias("p")
ranges = date_ranges.alias("r")

national_commodity_growth_df = (
    prices
    .join(
        ranges,
        (F.col("p.commodity") == F.col("r.commodity"))
        & (F.col("p.currency") == F.col("r.currency"))
        & (
            (F.col("p.price_date") == F.col("r.start_date"))
            | (F.col("p.price_date") == F.col("r.end_date"))
        ),
        "inner",
    )
    .groupBy(
        F.col("r.commodity").alias("commodity"),
        F.col("r.currency").alias("currency"),
        F.col("r.start_date").alias("start_date"),
        F.col("r.end_date").alias("end_date"),
    )
    .agg(
        F.avg(
            F.when(
                F.col("p.price_date") == F.col("r.start_date"),
                F.col("p.price"),
            )
        ).alias("start_price"),
        F.avg(
            F.when(
                F.col("p.price_date") == F.col("r.end_date"),
                F.col("p.price"),
            )
        ).alias("end_price"),
    )
    .withColumn(
        "price_increase",
        F.col("end_price") - F.col("start_price"),
    )
    .withColumn(
        "price_increase_pct",
        F.when(
            F.col("start_price") != 0,
            F.round(
                F.col("price_increase") / F.col("start_price") * 100,
                2,
            ),
        ),
    )
    .orderBy(F.desc("price_increase_pct"))
)

national_commodity_growth_df.toPandas()

,commodity,currency,start_date,end_date,start_price,end_price,price_increase,price_increase_pct
0,potatoes,PHP,2008-01-01,2025-09-01,19.150588,66.095000,46.944412,245.13
1,carrots,PHP,2008-01-01,2025-09-01,59.358750,178.184889,118.826139,200.18
2,cabbage,PHP,2008-01-01,2025-09-01,36.604000,108.897568,72.293568,197.50
3,tomatoes,PHP,2008-01-01,2025-09-01,39.464706,93.614416,54.149710,137.21
4,eggs,PHP,2008-01-01,2025-09-01,4.280000,9.334583,5.054583,118.10
5,rice,PHP,2007-01-01,2025-09-01,21.402857,39.591325,18.188468,84.98
6,beans,PHP,2012-05-01,2025-09-01,79.230000,111.924375,32.694375,41.27
7,onions,PHP,2008-01-01,2025-09-01,134.022000,171.960441,37.938441,28.31
8,garlic,PHP,2008-01-01,2025-09-01,165.556667,181.770976,16.214309,9.79


# Check Duplicates

In [12]:
duplicates = (
    market_prices_df
    .groupBy("geo_id", "price_date", "commodity")
    .count()
    .filter(F.col("count") > 1)
)

print(f"Duplicate records: {duplicates.count():,}")
duplicates.show()

Duplicate records: 0
+------+----------+---------+-----+
|geo_id|price_date|commodity|count|
+------+----------+---------+-----+
+------+----------+---------+-----+



# Validate Imputations

In [13]:
imputation_summary = (
    market_prices_df
    .groupBy("commodity")
    .agg(
        F.count("*").alias("total_rows"),
        F.count("price").alias("available_prices"),
        F.sum(
            F.col("was_imputed").cast("integer")
        ).alias("imputed_observations"),
    )
    .withColumn(
        "imputation_rate",
        F.round(
            F.col("imputed_observations") / F.col("total_rows") * 100,
            2,
        ),
    )
)

imputation_summary.orderBy("commodity").show(truncate=False)

+---------+----------+----------------+--------------------+---------------+
|commodity|total_rows|available_prices|imputed_observations|imputation_rate|
+---------+----------+----------------+--------------------+---------------+
|beans    |24743     |4271            |1470                |5.94           |
|cabbage  |24743     |6204            |2746                |11.1           |
|carrots  |24743     |6518            |2957                |11.95          |
|eggs     |24743     |6152            |83                  |0.34           |
|garlic   |24743     |3539            |127                 |0.51           |
|onions   |24743     |6287            |95                  |0.38           |
|potatoes |24743     |3110            |99                  |0.4            |
|rice     |24743     |7442            |58                  |0.23           |
|tomatoes |24743     |6982            |77                  |0.31           |
+---------+----------+----------------+--------------------+---------------+

# Assemble Tables

In [14]:
gold_tables = {
    "market_monthly_prices": market_prices_df,
    "regional_monthly_prices": regional_prices_df,
    "national_monthly_prices": national_prices_df,
    "national_commodity_growth": national_commodity_growth_df,
}

In [15]:
save_gold_tables(gold_tables)

Gold table saved: market_monthly_prices (local)
Gold table saved: regional_monthly_prices (local)
Gold table saved: national_monthly_prices (local)
Gold table saved: national_commodity_growth (local)


In [16]:
from presyo_ph.medallion.gold import load_gold_table

for name in gold_tables:
    df = load_gold_table(spark, name)

    print(f"\n{name}")
    print(f"Rows: {df.count():,}")
    df.printSchema()
    df.limit(5).toPandas()

Gold table loaded: market_monthly_prices (local)

market_monthly_prices
Rows: 222,687
root
 |-- geo_id: string (nullable = true)
 |-- mkt_name: string (nullable = true)
 |-- adm1_name: string (nullable = true)
 |-- adm2_name: string (nullable = true)
 |-- price_date: date (nullable = true)
 |-- currency: string (nullable = true)
 |-- commodity: string (nullable = true)
 |-- price: double (nullable = true)
 |-- original_price: double (nullable = true)
 |-- was_imputed: boolean (nullable = true)

Gold table loaded: regional_monthly_prices (local)

regional_monthly_prices
Rows: 23,862
root
 |-- adm1_name: string (nullable = true)
 |-- price_date: date (nullable = true)
 |-- commodity: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- avg_price: double (nullable = true)
 |-- median_price: double (nullable = true)
 |-- min_price: double (nullable = true)
 |-- max_price: double (nullable = true)
 |-- markets_reporting: long (nullable = true)
 |-- imputed_observations: lon